# 04 — MobileNetV2 Visual Classification

This notebook reproduces the image-based MobileNetV2 branch of:

> **A Dual Representation Framework for Malicious QR Code Detection Using Fused Feature Learning and Deep Visual Modeling**

It consumes the exact train, validation, and test metadata generated by
`01_dataset_preparation.ipynb`.

## Workflow

1. Load controlled dataset partitions.
2. Resolve repository-relative image paths.
3. Build TensorFlow input pipelines.
4. Train a frozen MobileNetV2 classification head.
5. Fine-tune the final portion of the backbone.
6. Evaluate validation and untouched test sets.
7. Save the trained model, metrics, predictions, curves, and figures.

Robustness experiments are intentionally separated into
`07_validation_and_reliability.ipynb`.

## Reproducibility Note

This notebook preserves the configuration found in the final MobileNetV2 source notebook:

- image size: 128 × 128;
- batch size: 256;
- frozen-head epochs: 4;
- fine-tuning epochs: 12;
- frozen-head learning rate: 1e-3;
- fine-tuning learning rate: 1e-5;
- final 50 backbone layers selected for fine-tuning;
- dropout: 0.20;
- threshold: 0.5.

These values should be treated as the source-of-truth configuration unless the model is rerun under a revised protocol.

## 1. TensorFlow Environment and Deterministic Configuration

In [ ]:
# ============================================================
# TENSORFLOW ENVIRONMENT + REPRODUCIBILITY
# Run this cell after a fresh runtime restart.
# ============================================================

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["PYTHONHASHSEED"] = "42"
os.environ["TF_DETERMINISTIC_OPS"] = "1"

import json
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    precision_recall_curve,
    auc
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception as exc:
    print("Deterministic operations could not be fully enabled:", exc)

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow version:", tf.__version__)
print("Available GPUs:", gpus)

if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass

# Mixed precision is useful on modern Colab GPUs.
if gpus:
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision policy:", mixed_precision.global_policy())

## 2. Configuration and Controlled Dataset Splits

Paths are repository-relative. The optional local image cache is disabled by default because it was designed for Google Colab acceleration.

In [ ]:
# ============================================================
# FAST CONFIGURATION + LOAD EXISTING CONTROLLED SPLITS
# ============================================================

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebooks" else CURRENT_DIR

PROCESSED_DATA_DIR = PROJECT_ROOT / "Data" / "processed"
MODELS_DIR = PROJECT_ROOT / "Models" / "mobilenetv2"
MNV2_DIR = PROJECT_ROOT / "Results" / "mobilenetv2"

for folder in [MODELS_DIR, MNV2_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = PROCESSED_DATA_DIR / "train.csv"
VAL_CSV   = PROCESSED_DATA_DIR / "val.csv"
TEST_CSV  = PROCESSED_DATA_DIR / "test.csv"

for path in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    if not path.exists():
        raise FileNotFoundError(f"Required split file not found: {path}")

train_df = pd.read_csv(TRAIN_CSV).copy()
val_df   = pd.read_csv(VAL_CSV).copy()
test_df  = pd.read_csv(TEST_CSV).copy()

PATH_COLUMN = "image_path" if "image_path" in train_df.columns else "image"
LABEL_COLUMN = "label"
LABEL_ID_COLUMN = "label_id"

label_map = {"benign": 0, "malicious": 1}

for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    if PATH_COLUMN not in df.columns:
        raise KeyError(f"{name} split has no image-path column.")
    if LABEL_COLUMN not in df.columns:
        raise KeyError(f"{name} split has no label column.")

    df[PATH_COLUMN] = df[PATH_COLUMN].astype(str).str.strip()
    df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(str).str.strip().str.lower()

    if LABEL_ID_COLUMN not in df.columns:
        df[LABEL_ID_COLUMN] = df[LABEL_COLUMN].map(label_map)

    if df[LABEL_ID_COLUMN].isna().any():
        bad = df.loc[df[LABEL_ID_COLUMN].isna(), LABEL_COLUMN].unique()
        raise ValueError(f"Unrecognized labels in {name}: {bad}")

    df[LABEL_ID_COLUMN] = df[LABEL_ID_COLUMN].astype(int)

# Fast controlled configuration for A100.
IMG_SIZE = 128
BATCH_SIZE = 256
HEAD_EPOCHS = 4
FINE_TUNE_EPOCHS = 12

# MobileNetV2 normally has 154 layers.
# Fine-tune only the final 50 layers; early generic filters remain frozen.
FINE_TUNE_AT = max(0, 154 - 50)

HEAD_LR = 1e-3
FINE_TUNE_LR = 1e-5

# Set True when paths point to mounted Google Drive.
# Images are copied once to local Colab storage, substantially reducing epoch time.
USE_LOCAL_IMAGE_CACHE = False
LOCAL_CACHE_ROOT = PROJECT_ROOT / "Data" / "cache" / "mobilenetv2"

BEST_MODEL_PATH = MODELS_DIR / "best_mobilenetv2_fast_strengthened.keras"
HEAD_HISTORY_PATH = MNV2_DIR / "head_training_history.csv"
FT_HISTORY_PATH = MNV2_DIR / "fine_tuning_history.csv"

print("Path column:", PATH_COLUMN)
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)
print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Frozen-stage epochs:", HEAD_EPOCHS)
print("Fine-tuning epochs:", FINE_TUNE_EPOCHS)
print("Fine-tune starting index:", FINE_TUNE_AT)

print("\nClass distributions:")
print("Train:\n", train_df[LABEL_COLUMN].value_counts())
print("Validation:\n", val_df[LABEL_COLUMN].value_counts())
print("Test:\n", test_df[LABEL_COLUMN].value_counts())

## Source-of-Truth Hyperparameters

The following settings are taken directly from the final MobileNetV2 experiment notebook and should be used when updating the manuscript.

| Hyperparameter | Frozen-head stage | Fine-tuning stage |
|---|---:|---:|
| Backbone | MobileNetV2 with ImageNet weights | Same backbone |
| Input image size | 128 × 128 × 3 | 128 × 128 × 3 |
| Batch size | 256 | 256 |
| Maximum epochs | 4 | 12 |
| Optimizer | Adam | Adam |
| Learning rate | 1 × 10⁻³ | 1 × 10⁻⁵ |
| Loss function | Binary cross-entropy | Binary cross-entropy |
| Output activation | Sigmoid | Sigmoid |
| Global pooling | GlobalAveragePooling2D | GlobalAveragePooling2D |
| Dropout rate | 0.20 | 0.20 |
| Backbone trainability | Fully frozen | Final 50 layers selected for unfreezing |
| Batch-normalization layers | Frozen with backbone | Kept frozen |
| Gradient clipping | Not applied | `clipnorm = 1.0` |
| Early stopping patience | 2 | 3 |
| Early stopping monitor | Validation loss | Validation loss |
| Learning-rate reduction | Not used | Factor 0.2, patience 2, minimum 1 × 10⁻⁷ |
| Classification threshold | 0.50 | 0.50 |
| Data preprocessing | MobileNetV2 `preprocess_input` | MobileNetV2 `preprocess_input` |
| Image resizing | Bilinear with antialiasing | Bilinear with antialiasing |
| Random seed | 42 | 42 |

No separate dense hidden layer or data-augmentation layer is used in this final implementation.

In [ ]:
MOBILENETV2_HYPERPARAMETERS = {
    "backbone": "MobileNetV2",
    "pretrained_weights": "ImageNet",
    "input_size": [128, 128, 3],
    "batch_size": 256,
    "frozen_head_epochs": 4,
    "fine_tuning_epochs": 12,
    "frozen_head_learning_rate": 1e-3,
    "fine_tuning_learning_rate": 1e-5,
    "loss_function": "BinaryCrossentropy",
    "output_activation": "sigmoid",
    "global_pooling": "GlobalAveragePooling2D",
    "dropout_rate": 0.20,
    "fine_tuned_backbone_layers": 50,
    "batch_normalization_trainable": False,
    "fine_tuning_clipnorm": 1.0,
    "frozen_head_early_stopping_patience": 2,
    "fine_tuning_early_stopping_patience": 3,
    "reduce_lr_factor": 0.2,
    "reduce_lr_patience": 2,
    "minimum_learning_rate": 1e-7,
    "classification_threshold": 0.50,
    "resize_method": "bilinear",
    "antialias": True,
    "random_seed": 42,
    "data_augmentation": False,
    "dense_hidden_layer": False,
}

HYPERPARAMETER_PATH = MNV2_DIR / "mobilenetv2_hyperparameters.json"

with HYPERPARAMETER_PATH.open("w", encoding="utf-8") as file:
    json.dump(MOBILENETV2_HYPERPARAMETERS, file, indent=2)

print("Saved hyperparameters:", HYPERPARAMETER_PATH)

## 3. Verify and Resolve Image Paths

In [ ]:
# ============================================================
# VERIFY PATHS + OPTIONAL ONE-TIME LOCAL IMAGE CACHE
# ============================================================

import shutil
from tqdm.auto import tqdm

def retain_existing_paths(df, split_name):
    exists = df[PATH_COLUMN].map(lambda path: (PROJECT_ROOT / path).exists())
    missing = int((~exists).sum())

    print(f"{split_name}: {exists.sum():,}/{len(df):,} paths exist")

    if missing:
        print("First missing paths:")
        print(df.loc[~exists, PATH_COLUMN].head(10).tolist())

    cleaned = df.loc[exists].reset_index(drop=True).copy()
    # TensorFlow receives absolute paths so execution works from either
    # the repository root or the Notebooks directory.
    cleaned[PATH_COLUMN] = cleaned[PATH_COLUMN].map(
        lambda path: str((PROJECT_ROOT / path).resolve())
    )

    if cleaned.empty:
        raise RuntimeError(f"No valid images remain in the {split_name} split.")

    return cleaned

train_df = retain_existing_paths(train_df, "Train")
val_df   = retain_existing_paths(val_df, "Validation")
test_df  = retain_existing_paths(test_df, "Test")


def cache_split_locally(df, split_name):
    """
    Copy each image once to Colab local storage.
    Unique indexed filenames avoid collisions between equal basenames.
    Existing files are reused when a cell is rerun.
    """
    split_dir = LOCAL_CACHE_ROOT / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    local_paths = []

    for row_index, source_path in tqdm(
        enumerate(df[PATH_COLUMN].astype(str).tolist()),
        total=len(df),
        desc=f"Caching {split_name}"
    ):
        source = PROJECT_ROOT / source_path
        suffix = source.suffix.lower() if source.suffix else ".png"
        destination = split_dir / f"{row_index:07d}{suffix}"

        if not destination.exists() or destination.stat().st_size == 0:
            shutil.copy2(source, destination)

        local_paths.append(str(destination))

    cached = df.copy()
    cached[PATH_COLUMN] = local_paths
    return cached


if USE_LOCAL_IMAGE_CACHE:
    print("\nCopying/reusing images in local Colab storage.")
    print("This one-time step can take a while, but subsequent training epochs should be much faster.")

    train_df = cache_split_locally(train_df, "train")
    val_df   = cache_split_locally(val_df, "validation")
    test_df  = cache_split_locally(test_df, "test")

    # Verify local cache.
    train_df = retain_existing_paths(train_df, "Local train")
    val_df   = retain_existing_paths(val_df, "Local validation")
    test_df  = retain_existing_paths(test_df, "Local test")

    train_df.to_csv(MNV2_DIR / "train_local_paths.csv", index=False)
    val_df.to_csv(MNV2_DIR / "validation_local_paths.csv", index=False)
    test_df.to_csv(MNV2_DIR / "test_local_paths.csv", index=False)

    print("Local cache root:", LOCAL_CACHE_ROOT)
else:
    print("Local image caching is disabled.")

## 4. TensorFlow Image Pipeline

In [ ]:
# ============================================================
# FAST TF.DATA IMAGE PIPELINES
# ============================================================

AUTOTUNE = tf.data.AUTOTUNE

def decode_resize_preprocess(path, label):
    raw = tf.io.read_file(path)
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE],
        method=tf.image.ResizeMethod.BILINEAR,
        antialias=True
    )

    image = tf.cast(image, tf.float32)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    label = tf.cast(label, tf.float32)
    return image, label


def make_dataset(df, training=False, cache=False):
    paths = df[PATH_COLUMN].astype(str).to_numpy()
    labels = df[LABEL_ID_COLUMN].astype("float32").to_numpy()

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    options = tf.data.Options()
    options.experimental_deterministic = not training
    ds = ds.with_options(options)

    if training:
        ds = ds.shuffle(
            buffer_size=min(len(df), 30000),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        decode_resize_preprocess,
        num_parallel_calls=AUTOTUNE,
        deterministic=not training
    )

    # Cache validation/test because they are unchanged across epochs.
    # The training set is intentionally not cached in RAM.
    if cache:
        ds = ds.cache()

    ds = ds.batch(BATCH_SIZE, drop_remainder=False)
    ds = ds.prefetch(AUTOTUNE)
    return ds


train_ds = make_dataset(train_df, training=True, cache=False)
val_ds   = make_dataset(val_df, training=False, cache=True)
test_ds  = make_dataset(test_df, training=False, cache=True)

sample_images, sample_labels = next(iter(train_ds))
print("Image batch shape:", sample_images.shape)
print("Label batch shape:", sample_labels.shape)
print("Image value range:", float(tf.reduce_min(sample_images)), float(tf.reduce_max(sample_images)))

## 5. Stage 1 — Frozen-Backbone Warm-Up

MobileNetV2 is initialized with ImageNet weights and used first as a fixed feature extractor.

In [ ]:
# ============================================================
# STAGE 1: SHORT FROZEN-BACKBONE WARM-UP
# ============================================================

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="qr_image")
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
x = tf.keras.layers.Dropout(0.20, seed=SEED, name="head_dropout")(x)
outputs = tf.keras.layers.Dense(
    1,
    activation="sigmoid",
    dtype="float32",
    name="prediction"
)(x)

model = tf.keras.Model(inputs, outputs, name="MobileNetV2_QR_Fast_Strengthened")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=HEAD_LR),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

head_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODELS_DIR / "best_mobilenetv2_frozen_head.keras"),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=2,
        min_delta=1e-4,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(str(HEAD_HISTORY_PATH), append=False),
    tf.keras.callbacks.TerminateOnNaN()
]

model.summary()

head_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=HEAD_EPOCHS,
    callbacks=head_callbacks,
    verbose=1
)

## 6. Stage 2 — QR-Specific Fine-Tuning

The final portion of the MobileNetV2 backbone is unfrozen while Batch Normalization layers remain frozen.

In [ ]:
# ============================================================
# STAGE 2: CONSERVATIVE QR-SPECIFIC FINE-TUNING
# ============================================================

base_model.trainable = True

for index, layer in enumerate(base_model.layers):
    if index < FINE_TUNE_AT:
        layer.trainable = False
    else:
        # Freeze Batch Normalization moving statistics during transfer learning.
        layer.trainable = not isinstance(layer, tf.keras.layers.BatchNormalization)

trainable_base_layers = sum(int(layer.trainable) for layer in base_model.layers)

print("Total base layers:", len(base_model.layers))
print("Fine-tune starting index:", FINE_TUNE_AT)
print("Trainable base layers:", trainable_base_layers)
print("Total trainable parameters:", sum(
    int(tf.keras.backend.count_params(weight))
    for weight in model.trainable_weights
))

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=FINE_TUNE_LR,
        clipnorm=1.0
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

fine_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        min_delta=1e-5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(str(FT_HISTORY_PATH), append=False),
    tf.keras.callbacks.TerminateOnNaN()
]

fine_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=fine_callbacks,
    verbose=1
)

if not BEST_MODEL_PATH.exists():
    model.save(BEST_MODEL_PATH)

print("Best model saved to:", BEST_MODEL_PATH)

## 7. Validation Evaluation

In [ ]:
# ============================================================
# VALIDATION EVALUATION
# ============================================================

best_model = tf.keras.models.load_model(BEST_MODEL_PATH)

def predict_dataset(model, dataset):
    start = time.perf_counter()
    probabilities = model.predict(dataset, verbose=1).reshape(-1)
    elapsed = time.perf_counter() - start
    return probabilities, elapsed

val_true = val_df[LABEL_ID_COLUMN].to_numpy(dtype=int)
val_proba, val_inference_seconds = predict_dataset(best_model, val_ds)
val_pred = (val_proba >= 0.5).astype(int)

val_metrics = {
    "split": "validation",
    "threshold": 0.5,
    "accuracy": accuracy_score(val_true, val_pred),
    "precision": precision_score(val_true, val_pred, zero_division=0),
    "recall": recall_score(val_true, val_pred, zero_division=0),
    "f1_score": f1_score(val_true, val_pred, zero_division=0),
    "roc_auc": roc_auc_score(val_true, val_proba),
    "average_precision": average_precision_score(val_true, val_proba),
    "inference_seconds": val_inference_seconds
}

pd.DataFrame([val_metrics]).to_csv(
    MNV2_DIR / "mobilenetv2_validation_metrics.csv",
    index=False
)

val_output = val_df[[PATH_COLUMN, LABEL_COLUMN, LABEL_ID_COLUMN]].copy()
val_output["y_true"] = val_true
val_output["y_pred"] = val_pred
val_output["y_proba"] = val_proba
val_output.to_csv(MNV2_DIR / "mobilenetv2_validation_predictions.csv", index=False)

print(pd.DataFrame([val_metrics]).T)
print("\nValidation classification report:")
print(classification_report(
    val_true,
    val_pred,
    target_names=["benign", "malicious"],
    digits=6,
    zero_division=0
))

## 8. Final Untouched Test Evaluation

The test set is evaluated only after model selection based on validation loss.

In [ ]:
# ============================================================
# FINAL TEST EVALUATION
# Test set is evaluated only after model selection on validation loss.
# ============================================================

test_true = test_df[LABEL_ID_COLUMN].to_numpy(dtype=int)
test_proba, test_inference_seconds = predict_dataset(best_model, test_ds)
test_pred = (test_proba >= 0.5).astype(int)

test_metrics = {
    "model": "MobileNetV2_Fast_Strengthened",
    "split": "test",
    "threshold": 0.5,
    "accuracy": accuracy_score(test_true, test_pred),
    "precision": precision_score(test_true, test_pred, zero_division=0),
    "recall": recall_score(test_true, test_pred, zero_division=0),
    "f1_score": f1_score(test_true, test_pred, zero_division=0),
    "roc_auc": roc_auc_score(test_true, test_proba),
    "average_precision": average_precision_score(test_true, test_proba),
    "inference_seconds": test_inference_seconds,
    "average_inference_seconds_per_image": test_inference_seconds / len(test_df)
}

test_metrics_df = pd.DataFrame([test_metrics])
test_metrics_df.to_csv(MNV2_DIR / "mobilenetv2_test_metrics.csv", index=False)

test_output = test_df[[PATH_COLUMN, LABEL_COLUMN, LABEL_ID_COLUMN]].copy()
test_output["y_true"] = test_true
test_output["y_pred"] = test_pred
test_output["y_proba"] = test_proba
test_output.to_csv(MNV2_DIR / "mobilenetv2_test_predictions.csv", index=False)

cm = confusion_matrix(test_true, test_pred)
pd.DataFrame(
    cm,
    index=["true_benign", "true_malicious"],
    columns=["pred_benign", "pred_malicious"]
).to_csv(MNV2_DIR / "mobilenetv2_test_confusion_matrix.csv")

print("=" * 68)
print("FAST STRENGTHENED MOBILENETV2 TEST RESULTS")
print("=" * 68)
for key, value in test_metrics.items():
    print(f"{key}: {value}")

print("\nClassification report:")
print(classification_report(
    test_true,
    test_pred,
    target_names=["benign", "malicious"],
    digits=6,
    zero_division=0
))

## 9. Training Curves

In [ ]:
# ============================================================
# TRAINING CURVES
# ============================================================

head_df = pd.read_csv(HEAD_HISTORY_PATH)
fine_df = pd.read_csv(FT_HISTORY_PATH)

# Offset fine-tuning epochs so both stages appear consecutively.
fine_df = fine_df.copy()
fine_df["epoch"] = fine_df["epoch"] + len(head_df)

combined_history = pd.concat(
    [
        head_df.assign(stage="frozen_head"),
        fine_df.assign(stage="fine_tuning")
    ],
    ignore_index=True
)
combined_history.to_csv(MNV2_DIR / "mobilenetv2_combined_history.csv", index=False)

plt.figure(figsize=(8, 5))
plt.plot(combined_history["epoch"], combined_history["loss"], label="Training Loss")
plt.plot(combined_history["epoch"], combined_history["val_loss"], label="Validation Loss")
plt.axvline(len(head_df) - 0.5, linestyle="--", label="Fine-tuning begins")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Fast Strengthened MobileNetV2 Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(MNV2_DIR / "mobilenetv2_loss.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(combined_history["epoch"], combined_history["accuracy"], label="Training Accuracy")
plt.plot(combined_history["epoch"], combined_history["val_accuracy"], label="Validation Accuracy")
plt.axvline(len(head_df) - 0.5, linestyle="--", label="Fine-tuning begins")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Fast Strengthened MobileNetV2 Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(MNV2_DIR / "mobilenetv2_accuracy.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Test Confusion Matrix, ROC Curve, and Precision–Recall Curve

In [ ]:
# ============================================================
# CONFUSION MATRIX, ROC CURVE, AND PRECISION-RECALL CURVE
# ============================================================

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Benign", "Malicious"]
)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, values_format="d")
ax.set_title("Fast Strengthened MobileNetV2 Test Confusion Matrix")
plt.tight_layout()
plt.savefig(MNV2_DIR / "mobilenetv2_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

fpr, tpr, _ = roc_curve(test_true, test_proba)
roc_auc_value = auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"Test ROC-AUC = {roc_auc_value:.6f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Fast Strengthened MobileNetV2 ROC Curve")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.savefig(MNV2_DIR / "mobilenetv2_roc_curve.png", dpi=300, bbox_inches="tight")
plt.show()

precision_values, recall_values, _ = precision_recall_curve(test_true, test_proba)
ap_value = average_precision_score(test_true, test_proba)

plt.figure(figsize=(7, 6))
plt.plot(recall_values, precision_values, label=f"Average Precision = {ap_value:.6f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Fast Strengthened MobileNetV2 Precision–Recall Curve")
plt.legend(loc="lower left")
plt.grid(True)
plt.tight_layout()
plt.savefig(MNV2_DIR / "mobilenetv2_pr_curve.png", dpi=300, bbox_inches="tight")
plt.show()

## 11. Final Summary

In [ ]:
validation_metrics_df = pd.DataFrame([val_metrics])
test_metrics_df = pd.DataFrame([test_metrics])

final_summary = pd.concat(
    [validation_metrics_df, test_metrics_df],
    ignore_index=True,
    sort=False,
)

FINAL_SUMMARY_PATH = MNV2_DIR / "mobilenetv2_final_summary.csv"
final_summary.to_csv(FINAL_SUMMARY_PATH, index=False)

print("=" * 72)
print("MOBILENETV2 TRAINING AND EVALUATION COMPLETED")
print("=" * 72)
display(final_summary)
print("Best model :", BEST_MODEL_PATH)
print("Results    :", MNV2_DIR)
print("Summary    :", FINAL_SUMMARY_PATH)
print("=" * 72)